In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3
import os

In [2]:
df = pd.read_csv("ai_bi_assistant_sales_data - GET.csv")

In [3]:
df

,Transaction_Id,Order_date,Product,Category,Region,Sales_channel,Quantity,Unit_price,Discount_amount,Status
0,TXN0001,03-01-2026,Laptop Stand,Accessories,North,Online,10,1200.0,0,Completed
1,TXN0002,05-01-2026,Wireless Mouse,Accessories,South,Retail,25,650.0,0,Completed
2,TXN0003,08-01-2026,Office Chair,Furniture,West,Online,4,7800.0,0,Completed
3,TXN0004,12-01-2026,Monitor,Electronics,East,Retail,6,9500.0,500,Completed
4,TXN0005,15-01-2026,Keyboard,Accessories,North,Online,18,900.0,0,Completed
5,TXN0006,18-01-2026,Desk,Furniture,South,Retail,3,12500.0,0,Completed
6,TXN0007,21-01-2026,USB-C Hub,Accessories,West,Online,12,1800.0,0,Completed
7,TXN0008,25-01-2026,Monitor,Electronics,North,Online,2,9500.0,0,Completed
8,TXN0009,02-02-2026,Laptop Stand,Accessories,East,Retail,14,1200.0,0,Completed
9,TXN0010,04-02-2026,Wireless Mouse,Accessories,West,Online,30,650.0,0,Completed


In [4]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
print("Dataset dimensions:", df.shape)


print("Column Names:")
print(df.columns.tolist())

Number of rows: 51
Number of columns: 10
Dataset dimensions: (51, 10)
Column Names:
['Transaction_Id', 'Order_date', 'Product', 'Category', 'Region', 'Sales_channel', 'Quantity', 'Unit_price', 'Discount_amount', 'Status']


In [5]:
print("\nData Types:")
print(df.dtypes)

missing_values = df.isnull().sum()
print("\nMissing values in each column:")
display(missing_values)

print("\nTotal duplicate rows:", df.duplicated().sum())


Data Types:
Transaction_Id      object
Order_date          object
Product             object
Category            object
Region              object
Sales_channel       object
Quantity             int64
Unit_price         float64
Discount_amount      int64
Status              object
dtype: object

Missing values in each column:


Transaction_Id     0
Order_date         0
Product            0
Category           0
Region             0
Sales_channel      0
Quantity           0
Unit_price         1
Discount_amount    0
Status             0
dtype: int64


Total duplicate rows: 0


In [6]:
df["Order_date"] = pd.to_datetime(
    df["Order_date"],
    format="%d-%m-%Y",
    errors="coerce"
)
print(df["Order_date"].head())

df['Unit_price'] = df['Unit_price'].fillna(
    df.groupby('Category')['Unit_price'].transform('first')
)
df['Unit_price']

0   2026-01-03
1   2026-01-05
2   2026-01-08
3   2026-01-12
4   2026-01-15
Name: Order_date, dtype: datetime64[ns]


0      1200.0
1       650.0
2      7800.0
3      9500.0
4       900.0
5     12500.0
6      1800.0
7      9500.0
8      1200.0
9       650.0
10     7800.0
11     9500.0
12      900.0
13    12500.0
14     1800.0
15      650.0
16     1200.0
17      650.0
18     7800.0
19     9500.0
20      900.0
21    12500.0
22     1800.0
23     9500.0
24     1200.0
25      650.0
26     7800.0
27     9500.0
28      900.0
29    12500.0
30     1800.0
31      650.0
32     9500.0
33      900.0
34     1200.0
35      650.0
36     7800.0
37     9500.0
38      900.0
39    12500.0
40     1800.0
41     9500.0
42     1200.0
43      650.0
44     7800.0
45     9500.0
46      900.0
47    12500.0
48     1800.0
49      900.0
50     1200.0
Name: Unit_price, dtype: float64

In [7]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51 entries, 0 to 50
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Transaction_Id   51 non-null     object        
 1   Order_date       51 non-null     datetime64[ns]
 2   Product          51 non-null     object        
 3   Category         51 non-null     object        
 4   Region           51 non-null     object        
 5   Sales_channel    51 non-null     object        
 6   Quantity         51 non-null     int64         
 7   Unit_price       51 non-null     float64       
 8   Discount_amount  51 non-null     int64         
 9   Status           51 non-null     object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(6)
memory usage: 4.1+ KB
None


In [8]:
categorical_columns = ['Product', 'Category', 'Region', 'Sales_channel', 'Status']

print("Categorical columns:")
print(categorical_columns)

for column in categorical_columns:
    print(f"\n{column}")
    print("Unique values:", df[column].nunique())
    print(df[column].unique())

Categorical columns:
['Product', 'Category', 'Region', 'Sales_channel', 'Status']

Product
Unique values: 7
['Laptop Stand' 'Wireless Mouse' 'Office Chair' 'Monitor' 'Keyboard'
 'Desk' 'USB-C Hub']

Category
Unique values: 3
['Accessories' 'Furniture' 'Electronics']

Region
Unique values: 4
['North' 'South' 'West' 'East']

Sales_channel
Unique values: 2
['Online' 'Retail']

Status
Unique values: 2
['Completed' 'Returned']


In [9]:
# Feature Engineering

df["Total_price"] = (
    df["Quantity"] * df["Unit_price"]
) - df["Discount_amount"]

df

,Transaction_Id,Order_date,Product,Category,Region,Sales_channel,Quantity,Unit_price,Discount_amount,Status,Total_price
0,TXN0001,2026-01-03,Laptop Stand,Accessories,North,Online,10,1200.0,0,Completed,12000.0
1,TXN0002,2026-01-05,Wireless Mouse,Accessories,South,Retail,25,650.0,0,Completed,16250.0
2,TXN0003,2026-01-08,Office Chair,Furniture,West,Online,4,7800.0,0,Completed,31200.0
3,TXN0004,2026-01-12,Monitor,Electronics,East,Retail,6,9500.0,500,Completed,56500.0
4,TXN0005,2026-01-15,Keyboard,Accessories,North,Online,18,900.0,0,Completed,16200.0
5,TXN0006,2026-01-18,Desk,Furniture,South,Retail,3,12500.0,0,Completed,37500.0
6,TXN0007,2026-01-21,USB-C Hub,Accessories,West,Online,12,1800.0,0,Completed,21600.0
7,TXN0008,2026-01-25,Monitor,Electronics,North,Online,2,9500.0,0,Completed,19000.0
8,TXN0009,2026-02-02,Laptop Stand,Accessories,East,Retail,14,1200.0,0,Completed,16800.0
9,TXN0010,2026-02-04,Wireless Mouse,Accessories,West,Online,30,650.0,0,Completed,19500.0


In [10]:
sales = df.copy()

sales = sales.drop_duplicates()
sales = sales.dropna()

sales = sales[
    (sales["Quantity"] > 0) &
    (sales["Unit_price"] >= 0)
]

In [11]:
os.makedirs("database", exist_ok=True)
conn = sqlite3.connect("database/sales.db")
sales.to_sql("sales", conn, if_exists="replace", index=False)

print("Database created successfully!")
print("Total valid records:", len(sales))  

print("\n", pd.read_sql_query("SELECT * FROM sales LIMIT 5", conn))

Database created successfully!
Total valid records: 50

   Transaction_Id           Order_date         Product     Category Region  \
0        TXN0001  2026-01-03 00:00:00    Laptop Stand  Accessories  North   
1        TXN0002  2026-01-05 00:00:00  Wireless Mouse  Accessories  South   
2        TXN0003  2026-01-08 00:00:00    Office Chair    Furniture   West   
3        TXN0004  2026-01-12 00:00:00         Monitor  Electronics   East   
4        TXN0005  2026-01-15 00:00:00        Keyboard  Accessories  North   

  Sales_channel  Quantity  Unit_price  Discount_amount     Status  Total_price  
0        Online        10      1200.0                0  Completed      12000.0  
1        Retail        25       650.0                0  Completed      16250.0  
2        Online         4      7800.0                0  Completed      31200.0  
3        Retail         6      9500.0              500  Completed      56500.0  
4        Online        18       900.0                0  Completed      1620

In [12]:
# Total Revenue
total_revenue = pd.read_sql_query("""
    SELECT SUM(Total_price) AS total_revenue
    FROM sales
    WHERE status = 'Completed'
""", conn)

# Product with Highest Revenue
top_product = pd.read_sql_query("""
    SELECT product, SUM(Total_price) AS total_revenue
    FROM sales
    WHERE status = 'Completed'
    GROUP BY product
    ORDER BY total_revenue DESC
    LIMIT 1
""", conn)

# Monthly Revenue Trend
monthly_revenue = pd.read_sql_query("""
    SELECT
        strftime('%Y-%m', order_date) AS month,
        SUM(Total_price) AS total_revenue
    FROM sales
    WHERE status = 'Completed'
    GROUP BY month
    ORDER BY month
""", conn)

# Region with Highest Sales
top_region = pd.read_sql_query("""
    SELECT region, SUM(Total_price) AS total_revenue
    FROM sales
    WHERE status = 'Completed'
    GROUP BY region
    ORDER BY total_revenue DESC
    LIMIT 1
""", conn)

# Revenue Comparison Between Two Months
month_comparison = pd.read_sql_query("""
    WITH monthly AS (
        SELECT
            strftime('%Y-%m', order_date) AS month,
            SUM(Total_price) AS revenue
        FROM sales
        WHERE status = 'Completed'
        GROUP BY month
    )
    SELECT
        month,
        revenue
    FROM monthly
    ORDER BY month DESC
    LIMIT 2
""", conn)


print("TOTAL REVENUE:")
display(total_revenue)

print("HIGHEST REVENUE PRODUCT:")
display(top_product)

print("MONTHLY REVENUE TREND:")
display(monthly_revenue)

print("HIGHEST SALES REGION:")
display(top_region)
print("REVENUE COMPARISON (LATEST TWO MONTHS):")
if len(month_comparison) == 2:
    latest = month_comparison.iloc[0]
    previous = month_comparison.iloc[1]

    change = latest["revenue"] - previous["revenue"]
    percentage = (
        (change / previous["revenue"]) * 100
        if previous["revenue"] != 0 else None
    )

    print("Latest month:", latest["month"])
    print("Previous month:", previous["month"])
    print("Revenue change:", round(change, 2))
    print("Percentage change:",
          round(percentage, 2) if percentage is not None else "Undefined")
else:
    print("Not enough months available for comparison.")

TOTAL REVENUE:


,total_revenue
0,1445200.0


HIGHEST REVENUE PRODUCT:


,Product,total_revenue
0,Monitor,473000.0


MONTHLY REVENUE TREND:


,month,total_revenue
0,2026-01,210250.0
1,2026-02,234600.0
2,2026-03,223900.0
3,2026-04,240700.0
4,2026-05,318500.0
5,2026-06,217250.0


HIGHEST SALES REGION:


,Region,total_revenue
0,South,411200.0


REVENUE COMPARISON (LATEST TWO MONTHS):
Latest month: 2026-06
Previous month: 2026-05
Revenue change: -101250.0
Percentage change: -31.79
